# Solution F1 · Which scaler for which model, with and without outliers

**Type**: Compare

## Objective

Compare five preprocessing choices (none, Standard, MinMax, Robust, Quantile) on four models (KNN, logistic
regression, tree, SVM) first on clean data and then with a small number of severe outliers, and answer with numbers "which scaler
is needed by whom".

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer, RobustScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
X = df[NUM].to_numpy()
y = df["defaulted"].to_numpy()
X_out = X.copy()
outlier_rows = np.random.default_rng(0).choice(len(X), 30, replace=False)
X_out[np.ix_(outlier_rows, [1, 3])] *= 100                 # unit error on income and amount
cv = StratifiedKFold(5, shuffle=True, random_state=0)

MODELS = {"knn": lambda: KNeighborsClassifier(15), "logistic": lambda: LogisticRegression(max_iter=2000),
          "tree": lambda: DecisionTreeClassifier(max_depth=5, random_state=0), "svm": lambda: SVC(C=1.0)}
SCALERS = {"none": lambda: "passthrough", "standard": lambda: StandardScaler(), "minmax": lambda: MinMaxScaler(),
           "robust": lambda: RobustScaler(), "quantile": lambda: QuantileTransformer(n_quantiles=200, output_distribution="normal")}

## Commented solution

In [ ]:
def evaluate_all(X, y):
    tab = pd.DataFrame(index=list(SCALERS), columns=list(MODELS), dtype=float)
    for scaler_name, make_scaler in SCALERS.items():
        for model_name, make_model in MODELS.items():
            pipe = make_pipeline(make_scaler(), make_model())
            tab.loc[scaler_name, model_name] = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()
    return tab

In [ ]:
clean_table = evaluate_all(X, y)
outlier_table = evaluate_all(X_out, y)
print("AUC with clean data"); display(clean_table.round(3))
print("AUC with 2% of the rows having a unit error"); display(outlier_table.round(3))

In [ ]:
tree_invariant = bool(np.ptp(clean_table["tree"]) < 0.01 and np.ptp(outlier_table["tree"]) < 0.01)
outlier_loss = clean_table - outlier_table
sensitive = outlier_loss[["knn", "logistic", "svm"]].mean(axis=1)
most_fragile_scaler = sensitive.idxmax()
most_robust_scaler = sensitive.idxmin()
best_knn_out = outlier_table["knn"].idxmax()
print("mean loss (scale-sensitive models):", sensitive.round(3).to_dict())
print("most fragile:", most_fragile_scaler, "| most robust:", most_robust_scaler, "| best scaler for the KNN with outliers:", best_knn_out)

## Check

In [ ]:
with verify():
    check(clean_table.shape == (5, 4) and outlier_table.shape == (5, 4), "two tables of 5 scalers x 4 models", "rows: scalers, columns: models")
    check(bool(tree_invariant) and np.ptp(clean_table["tree"]) < 0.01 and np.ptp(outlier_table["tree"]) < 0.01,
          "the tree does not depend on the scaler (variation below 0.01 in both tables)", "monotonic transformations do not change the splits")
    check(abs(clean_table.loc["standard", "knn"] - clean_table.loc["none", "knn"]) < 0.02,
          "on clean data scaling does not change the KNN: the informative features are also the ones on the larger scale",
          "on clean data KNN with and without StandardScaler should be close (less than 0.02)")
    check(outlier_table.loc["standard", "knn"] < outlier_table.loc["none", "knn"] - 0.05,
          "with outliers StandardScaler makes the KNN WORSE than not scaling at all",
          "with outliers StandardScaler should be worse than no scaler for the KNN (by more than 0.05)")
    check(outlier_table.loc["minmax", "knn"] < clean_table.loc["minmax", "knn"] - 0.03,
          "with outliers MinMax makes the KNN lose more than 0.03 of AUC", "outliers squash all the normal data with MinMax")
    check(most_fragile_scaler in ("minmax", "standard"), "the most fragile scaler is based on minimum/maximum or mean/deviation", "the most fragile is the one with the largest mean loss")
    check(most_robust_scaler in ("robust", "quantile"), "the most robust scaler is based on quantiles", "the most robust is robust or quantile")
    check(best_knn_out in ("robust", "quantile"), "with outliers the best scaler for the KNN is based on quantiles", "look at the knn column of outlier_table")
done()

## What you learned

- **Tree**: nothing changes, with any scaler and with or without outliers (AUC between 0.715 and 0.719). Scaling is wasted time.
- **On clean data** scaling makes no difference for the KNN (0.756 against 0.755): here the most informative features (income and
  amount) are also the ones with the larger scale, so the raw distance favors them by chance. Scaling is not
  always an improvement: it is a **protection** against the cases where the scale does not coincide with importance.
- **With outliers** the scalers based on mean or minimum/maximum become harmful: for the KNN StandardScaler drops to 0.657 and
  MinMaxScaler to 0.617, worse than not scaling (0.751); the logistic goes from 0.78 to 0.64 and 0.62. The normal data get squashed
  into a tiny interval and the distances between customers become almost equal.
- **RobustScaler and QuantileTransformer** use quantiles and resist: the QuantileTransformer reaches 0.758 (KNN) and 0.788
  (logistic) even with outliers. At the cost of losing the magnitude of the outlier, and with the extrapolation limits seen in B3.

## Common pitfall

Comparing scalers on clean data and concluding that "they are the same", or that scaling is not needed. The differences show up with real
data, which has outliers. When in doubt you measure in cross-validation, you don't choose out of habit.

## Interview question

> *Click on a question to show its answer.*

<details><summary>You have a dataset with outliers and a KNN. Which preprocessing do you propose, and how would you validate it?</summary>

I would start from a quantile-based scaler (`RobustScaler`, or `QuantileTransformer` if the distribution is very irregular),
because minimum/maximum and mean/standard deviation get dragged by the outliers. I would compare it with `StandardScaler` in cross-validation inside
a Pipeline, for the final model, and I would also check whether the outliers are errors (to correct) or real cases (to keep).

</details>